# R2-07 · Evaluación y validación de modelos — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea B · *Data Scientist* · Semana 12–13 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Elegir la métrica correcta (precision/recall/F1) cuando las clases están desbalanceadas.
2. Detectar y eliminar la **fuga de datos**.
3. Validar con **validación cruzada** y leer la **calibración** de las probabilidades.
4. Auditar el **fairness** del modelo entre grupos (regiones).

**Competencia de salida:** evaluar un clasificador con rigor: métricas honestas, sin fuga, validado y auditado por fairness.

**Dato real:** cantidad, monto y tamaño del proveedor de compras públicas de alimentos (ChileCompra).

**Entregable:** que las **5 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, brier_score_loss)

# "En vivo o caché": usa el archivo local; si falta (ej. Colab), lo baja del repo.
CSV = "compras_ml.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/compras_ml.csv", CSV)

df = pd.read_csv(CSV)
print("Datos cargados:", df.shape, "filas x columnas")
df.head()

### Preparación de datos

In [ ]:
# Objetivo: predecir si una orden la adjudica un proveedor GRANDE.
y = (df["tamano_proveedor"] == "Grande").astype(int)
NUM = ["cantidad", "monto_total"]
CAT = ["categoria", "region_comprador"]
X = df[NUM + CAT]

pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT)],
                        remainder="passthrough")
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print("Positivos (Grande):", round(100*y.mean(), 1), "% — clase desbalanceada")

## 1. Métricas más allá de la *accuracy*

Con clases desbalanceadas la **exactitud** engaña: un modelo que siempre dice «no Grande» acierta mucho y no sirve. Por eso miramos **precision**, **recall** y **F1**.

### ✍️ Ejercicio 1 — Entrena un modelo limpio y calcula las 4 métricas

In [ ]:
modelo = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])
modelo.fit(Xtr, ytr)
pred = modelo.predict(Xte)
metricas = {
    "accuracy":  accuracy_score(yte, pred),
    "precision": precision_score(yte, pred, zero_division=0),
    "recall":    recall_score(yte, pred, zero_division=0),
    "f1":        f1_score(yte, pred, zero_division=0),
}
print(metricas)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert set(metricas) == {"accuracy", "precision", "recall", "f1"}
    assert all(0.0 <= v <= 1.0 for v in metricas.values())
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'Usa accuracy_score, precision_score, recall_score y f1_score sobre (yte, pred).')
    print("   Detalle:", e)

## 2. Fuga de datos (*data leakage*)

Una **fuga** es usar información que no tendrías al predecir de verdad. Aquí `tamano_num` **codifica el tamaño del proveedor** → es casi la respuesta. Compara qué pasa al incluirla.

### ✍️ Ejercicio 2 — Compara la accuracy CON fuga vs SIN fuga

In [ ]:
Xtr_f = Xtr.assign(tamano_num=df.loc[Xtr.index, "tamano_num"])
Xte_f = Xte.assign(tamano_num=df.loc[Xte.index, "tamano_num"])
pre_f = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT)], remainder="passthrough")
m_fuga = Pipeline([("pre", pre_f), ("clf", DecisionTreeClassifier(random_state=42))]).fit(Xtr_f, ytr)
acc_con_fuga = accuracy_score(yte, m_fuga.predict(Xte_f))
acc_sin_fuga = accuracy_score(yte, modelo.predict(Xte))
print("CON fuga:", round(acc_con_fuga, 3), "| SIN fuga:", round(acc_sin_fuga, 3))

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert acc_con_fuga > 0.99, "Con la fuga la accuracy deberia ser casi perfecta"
    assert acc_sin_fuga < acc_con_fuga, "El modelo honesto rinde menos que el que hace trampa"
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'tamano_num distingue Grande casi perfecto: acc_con_fuga ~1.0; el modelo limpio rinde menos.')
    print("   Detalle:", e)

## 3. Validación cruzada

Un solo split puede mentir por suerte. La **validación cruzada** entrena y evalúa en varios cortes y promedia, dando una estimación más confiable.

### ✍️ Ejercicio 3 — 5-fold cross-validation del modelo limpio

In [ ]:
scores = cross_val_score(modelo, X, y, cv=5, scoring="f1")
print("F1 por fold:", scores.round(3), "| media:", round(scores.mean(), 3))

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert len(scores) == 5
    assert 0.0 <= scores.mean() <= 1.0
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "cross_val_score(modelo, X, y, cv=5, scoring='f1') devuelve un array de 5 valores.")
    print("   Detalle:", e)

## 4. Calibración

¿Las probabilidades del modelo son creíbles? El **Brier score** las mide (0 = perfectas). Además se pueden **ver** en una curva de calibración.

### ✍️ Ejercicio 4 — Brier score de las probabilidades

In [ ]:
proba = modelo.predict_proba(Xte)[:, 1]
brier = brier_score_loss(yte, proba)
print("Brier score:", round(brier, 4))

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert 0.0 <= brier <= 1.0
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'predict_proba(Xte)[:, 1] da la prob. de la clase positiva; pasala a brier_score_loss(yte, proba).')
    print("   Detalle:", e)

In [ ]:
# (ilustración) Curva de calibración: ¿la probabilidad predicha coincide con la realidad?
frac_pos, mean_pred = calibration_curve(yte, proba, n_bins=10)
plt.figure(figsize=(5, 4))
plt.plot([0, 1], [0, 1], "--", color="gray", label="perfecta")
plt.plot(mean_pred, frac_pos, "o-", color="#4240e5", label="modelo")
plt.xlabel("Probabilidad predicha"); plt.ylabel("Frecuencia real observada")
plt.title("Curva de calibración"); plt.legend(); plt.show()

## 5. *Fairness* por grupo

Un buen promedio puede esconder un mal desempeño en un grupo. Mide el **recall por región**: si el modelo detecta proveedores grandes mucho mejor en unas regiones que en otras, hay un sesgo a revisar.

### ✍️ Ejercicio 5 — Recall por región y su disparidad

In [ ]:
reg_te = df.loc[Xte.index, "region_comprador"]
recall_por_region = {}
for r in reg_te.unique():
    m = (reg_te == r).values
    if m.sum() >= 20 and yte[m].sum() > 0:      # solo grupos con datos suficientes
        recall_por_region[r] = recall_score(yte[m], pred[m], zero_division=0)
disparidad = max(recall_por_region.values()) - min(recall_por_region.values())
print("Disparidad de recall entre regiones:", round(disparidad, 3))

In [ ]:
# ── Celda de chequeo · Ejercicio 5 ──────────────────────────────────────────
try:
    assert len(recall_por_region) >= 2
    assert disparidad >= 0.0
    print("✅ Ejercicio 5: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'recall_score(yte[m], pred[m], zero_division=0) por grupo; disparidad = max(...) - min(...).')
    print("   Detalle:", e)

In [ ]:
# (ilustración) Recall por región: ¿el modelo es parejo en todo el país?
items = sorted(recall_por_region.items(), key=lambda kv: kv[1])
plt.figure(figsize=(6, 4))
plt.barh([k for k, _ in items], [v for _, v in items], color="#4240e5")
plt.xlabel("Recall (clase Grande)"); plt.title("Fairness: recall por región")
plt.tight_layout(); plt.show()

## Cierre

- La **accuracy** sola engaña con clases desbalanceadas: mira precision/recall/F1.
- La **fuga de datos** infla los resultados — desconfía de un modelo "demasiado perfecto".
- **Cross-validation** > un solo split. La **calibración** dice si las probabilidades son creíbles.
- El **fairness** se mide por grupo: un buen promedio puede esconder un mal desempeño en una región.

> *Ética distribuida: evaluar bien es parte de un uso responsable de datos en el Estado.*